**Multi-Attention & Transformer Comparisons:**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -o "/content/drive/MyDrive/Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip" -d /content/data
!unzip -o "/content/data/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip" -d /content/data
!unzip -o "/content/data/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip" -d /content/data


Mounted at /content/drive
Archive:  /content/drive/MyDrive/Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip
  inflating: /content/data/Basics of BERT and XLM-RoBERTa - PyTorch/sample_submission.csv  
 extracting: /content/data/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip  
 extracting: /content/data/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip  
Archive:  /content/data/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip
  inflating: /content/data/train.csv  
Archive:  /content/data/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip
  inflating: /content/data/test.csv  


In [3]:
import pandas as pd

train_df = pd.read_csv('/content/data/train.csv')
test_df = pd.read_csv('/content/data/test.csv')

train_df.head()

,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


In [4]:
train_df = train_df[train_df['language'] == 'English']
train_df.head()

,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
7,fdcd1bd867,From Cockpit Country to St. Ann's Bay,From St. Ann's Bay to Cockpit Country.,en,English,2
8,7cfb3d272c,"Look, it's your skin, but you're going to be i...",The boss will fire you if he sees you slacking...,en,English,1


In [28]:
# Single-Head Attention Implementation

import torch
import torch.nn as nn
import math

class SingleHeadAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Linear projections for Q, K, V
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        # x shape: (batch_size, seq_len, hidden_dim)

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        # Attention scores: Q @ K.T
        scores = torch.matmul(Q, K.transpose(-2, -1))

        # Scale scores
        scores = scores / math.sqrt(self.hidden_dim)

        # Convert scores into probabilities
        attention_weights = torch.softmax(scores, dim=-1)

        # Weighted sum of values
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

In [29]:
batch_size = 2
seq_len = 5
hidden_dim = 16

x = torch.randn(batch_size, seq_len, hidden_dim)

attention = SingleHeadAttention(hidden_dim)

output, attention_weights = attention(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", attention_weights.shape)

print("\nAttention weights:")
print(attention_weights)

Input shape: torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])
Attention weights shape: torch.Size([2, 5, 5])

Attention weights:
tensor([[[0.1707, 0.2947, 0.2662, 0.1656, 0.1028],
         [0.1503, 0.2448, 0.2249, 0.1835, 0.1965],
         [0.3023, 0.1520, 0.2006, 0.1233, 0.2219],
         [0.1496, 0.2499, 0.2274, 0.1943, 0.1788],
         [0.2258, 0.1458, 0.1979, 0.2275, 0.2029]],

        [[0.3404, 0.1016, 0.1836, 0.2025, 0.1719],
         [0.1655, 0.3244, 0.1813, 0.1108, 0.2179],
         [0.1289, 0.3094, 0.2208, 0.1059, 0.2349],
         [0.0967, 0.2666, 0.1556, 0.2136, 0.2676],
         [0.1638, 0.2961, 0.2046, 0.1803, 0.1551]]],
       grad_fn=<SoftmaxBackward0>)


In [30]:
# Multi-Head Attention Module:

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.1):
        super().__init__()

        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)

        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        # x shape: (batch_size, seq_len, hidden_dim)
        residual = x

        batch_size, seq_len, hidden_dim = x.shape

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        # Split hidden_dim into num_heads
        # from: (batch_size, seq_len, hidden_dim)
        # to:   (batch_size, num_heads, seq_len, head_dim)
        Q = Q.reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        attention_output = torch.matmul(attention_weights, V)

        # Concatenate heads
        # from: (batch_size, num_heads, seq_len, head_dim)
        # to:   (batch_size, seq_len, hidden_dim)
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.reshape(batch_size, seq_len, hidden_dim)

        output = self.out_proj(attention_output)

        # Residual connection + LayerNorm
        output = self.norm(residual + self.dropout(output))

        return output, attention_weights

In [31]:
batch_size = 2
seq_len = 5
hidden_dim = 32
num_heads = 4

x = torch.randn(batch_size, seq_len, hidden_dim)

mha = MultiHeadAttention(
    hidden_dim=hidden_dim,
    num_heads=num_heads,
    dropout=0.1
)

output, attention_weights = mha(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", attention_weights.shape)

Input shape: torch.Size([2, 5, 32])
Output shape: torch.Size([2, 5, 32])
Attention weights shape: torch.Size([2, 4, 5, 5])


In [32]:
# Custom Encoder Stack & Training Loop:

class EncoderBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attention = MultiHeadAttention(hidden_dim, num_heads, dropout)

        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, hidden_dim)
        )

        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x, attention_weights = self.attention(x)

        ff_output = self.ff(x)

        x = self.norm(x + self.dropout(ff_output))

        return x, attention_weights

In [35]:
class CustomEncoderClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        hidden_dim,
        num_heads,
        ff_dim,
        num_layers,
        num_classes,
        max_len=128,
        dropout=0.1
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.position_embedding = nn.Embedding(max_len, hidden_dim)

        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(hidden_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        positions = torch.arange(seq_len, device=input_ids.device)
        positions = positions.unsqueeze(0).expand(batch_size, seq_len)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)

        all_attention_weights = []

        for block in self.encoder_blocks:
            x, attention_weights = block(x)
            all_attention_weights.append(attention_weights)

        pooled = x[:, 0, :]   # CLS token

        logits = self.classifier(pooled)

        return logits, all_attention_weights

In [36]:
from sklearn.model_selection import train_test_split
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import torch

In [37]:
train_df = train_df[train_df["language"] == "English"].copy()

In [38]:
train_df["text"] = "[CLS] " + train_df["premise"] + " [SEP] " + train_df["hypothesis"]

In [39]:
train_texts, val_texts, train_labels, val_labels = train_test_split(train_df["text"], train_df["label"], test_size=0.1, random_state=42, stratify=train_df["label"])

In [40]:
def tokenize(text):
    return text.lower().split()

In [51]:
counter = Counter()

for text in train_texts:
    counter.update(tokenize(text))

vocab = {"<PAD>": 0, "<UNK>": 1, "[cls]": 2, "[sep]": 3}

for word, _ in counter.most_common(10000):
    if word not in vocab:
        vocab[word] = len(vocab)

print(encode("[cls] hello world [sep] test")[:10])

tensor([   2, 4850,  253,    3,  564,    0,    0,    0,    0,    0])


In [42]:
MAX_LEN = 128

def encode(text):
    tokens = tokenize(text)

    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    ids = ids[:MAX_LEN]

    if len(ids) < MAX_LEN:
        ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))

    return torch.tensor(ids, dtype=torch.long)

In [43]:
class NLIDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "input_ids": encode(self.texts[idx]),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [44]:
train_dataset = NLIDataset(train_texts, train_labels)
val_dataset = NLIDataset(val_texts, val_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomEncoderClassifier(vocab_size=len(vocab), hidden_dim=128, num_heads=4, ff_dim=256, num_layers=2, num_classes=3, max_len=MAX_LEN, dropout=0.1).to(device)

model = CustomEncoderClassifier(vocab_size=len(vocab), hidden_dim=256, num_heads=8, ff_dim=512, num_layers=3, num_classes=3)

In [46]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

In [47]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits, _ = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [48]:
def evaluate(model, loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits, _ = model(input_ids)

            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [52]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Training loss: {train_loss:.4f}")
    print(f"Validation accuracy: {val_acc:.4f}")
    print("-" * 30)

Epoch 1/10
Training loss: 1.1057
Validation accuracy: 0.3814
------------------------------
Epoch 2/10
Training loss: 1.0861
Validation accuracy: 0.3639
------------------------------
Epoch 3/10
Training loss: 1.0780
Validation accuracy: 0.3595
------------------------------
Epoch 4/10
Training loss: 1.0586
Validation accuracy: 0.4178
------------------------------
Epoch 5/10
Training loss: 1.0381
Validation accuracy: 0.3668
------------------------------
Epoch 6/10
Training loss: 1.0057
Validation accuracy: 0.3814
------------------------------
Epoch 7/10
Training loss: 0.9724
Validation accuracy: 0.3770
------------------------------
Epoch 8/10
Training loss: 0.9240
Validation accuracy: 0.3755
------------------------------
Epoch 9/10
Training loss: 0.8632
Validation accuracy: 0.3508
------------------------------
Epoch 10/10
Training loss: 0.8097
Validation accuracy: 0.3552
------------------------------


The custom attention encoder showed decreasing training loss, which means the model was learning patterns from the training data. However, validation accuracy stayed around 0.35-0.42 and was unstable. This suggests that the lightweight model had limited generalization ability. The main reasons are simple tokenization, training from scratch, a small architecture and no large-scale pretraining.